In [1]:
#Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

In [2]:
# Loading the data 
df = pd.read_csv(r"D:\NTI-ML\nti-projects\employee-attrition-analysis\data\raw\employee_attrition_course.csv")
df.head()  

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,RecordID,ManagerNotes
0,41.0,Yes,Travel_Rarely,1102,sales,1,2,Life Sciences,1,1,2,Female,94,3,2,Sales Executive,4,Single,NaN,19479,8,Y,Yes,11,3,1,80,0,8.0,0,1,6.0,4,0,5,100001,NaN
1,49.0,No,Travel_Frequently,279,Research and Development,8,1,Life Sciences,1,2,3,Male,61,2,2,Research Scientist,2,Married,NaN,24907,1,Y,No,23,4,4,80,1,10.0,3,3,10.0,7,1,7,100002,NaN
2,37.0,Yes,Travel_Rarely,1373,Research and Development,2,2,Other,1,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090.0,2396,6,Y,Yes,15,3,2,80,0,7.0,3,3,0.0,0,0,0,100003,Needs Improvement
3,33.0,No,Travel_Frequently,1392,Research and Development,3,4,Life Sciences,1,5,4,Female,56,3,1,Research Scientist,3,Married,2909.0,23159,1,Y,Yes,11,3,3,80,0,8.0,3,3,NaN,7,3,0,100004,Average
4,27.0,No,Travel_Rarely,591,Research and Development,2,1,Medical,1,7,1,Male,40,3,1,Laboratory Technician,2,Married,3468.0,16632,9,Y,No,12,3,4,80,1,6.0,3,3,2.0,2,2,2,100005,NaN


In [3]:
df_clean = df.copy() # Get a copy of the dat 

# Standardize categorical text values
# Remove leading/trailing whitespace and normalize text formatting
for col in ['Department', 'Gender', 'JobRole', 'BusinessTravel']:
    df_clean[col] = df_clean[col].astype(str).str.strip()

# Standardize Gender values
df_clean['Gender'] = df_clean['Gender'].str.title()

# Standardize BusinessTravel categories
# Merge equivalent categories with different spellings or formatting
business_travel_map = {
    'Travel_Rarely': 'Travel_Rarely',
    'Travel Rarely': 'Travel_Rarely',
    'Travel-Rarely': 'Travel_Rarely',
    'TRAVEL_RARELY': 'Travel_Rarely',
    'travel_rarely': 'Travel_Rarely',
    'Travel_Frequently': 'Travel_Frequently',
    'Non-Travel': 'Non-Travel'
}
df_clean['BusinessTravel'] = df_clean['BusinessTravel'].replace(business_travel_map)

# Correct inconsistent Department values
department_map = {
    'sales': 'Sales',
    'Human Resourses': 'Human Resources'
}
df_clean['Department'] = df_clean['Department'].replace(department_map)

# Standardize JobRole values
# Correct spelling variations and abbreviations
jobrole_map = {
    'Sales Ex.': 'Sales Executive',
    'Sales Executve': 'Sales Executive',
    'sales executive': 'Sales Executive'
}
df_clean['JobRole'] = df_clean['JobRole'].replace(jobrole_map)

print("Categorical values have been standardized successfully.")

df_clean['Department'].value_counts()

Categorical values have been standardized successfully.


Department
Research and Development    973
Sales                       456
Human Resources              66
Name: count, dtype: int64

> **Observation:** Inconsistent categorical values were standardized by removing extra whitespace, normalizing text formatting, correcting spelling errors, and merging equivalent category labels. This ensures that identical categories are represented consistently throughout the dataset, improving data quality and preventing duplicate categories during analysis and modeling.

In [4]:
# Verify that categorical values have been standardized correctly
print(df_clean['BusinessTravel'].value_counts())
print()

print(df_clean['Gender'].value_counts())
print()

print(df_clean['JobRole'].value_counts())

BusinessTravel
Travel_Rarely        1063
Travel_Frequently     281
Non-Travel            151
Name: count, dtype: int64

Gender
Male      897
Female    598
Name: count, dtype: int64

JobRole
Sales Executive              329
Research Scientist           290
Laboratory Technician        259
Manufacturing Director       146
Healthcare Representative    131
Manager                      105
Sales Representative          87
Research Director             80
Human Resources               53
nan                           15
Name: count, dtype: int64


In [5]:
# Remove constant (zero-variance) features
# These columns contain a single value across all records and provide no analytical or predictive information.
constant_cols = ['EmployeeCount', 'Over18', 'StandardHours']

df_clean = df_clean.drop(columns=constant_cols)

print("Removed constant columns:", constant_cols)

Removed constant columns: ['EmployeeCount', 'Over18', 'StandardHours']


In [6]:
# Remove the ManagerNotes column
# The column contains a high proportion of missing values and unstructured free-text,
# making it unsuitable for the current structured data analysis.

df_clean = df_clean.drop(columns=['ManagerNotes'])

print("Removed column: ManagerNotes")

Removed column: ManagerNotes


In [7]:
# Remove duplicated records while ignoring RecordID
# RecordID is a unique identifier and should not be considered when detecting duplicate observations.

before = len(df_clean)

df_clean = df_clean.drop_duplicates(
    subset=[col for col in df_clean.columns if col != 'RecordID']
)

after = len(df_clean)

print(f"Removed {before - after} duplicated records.")
print(f"Dataset size after removal: {after}")

Removed 25 duplicated records.
Dataset size after removal: 1470


In [8]:
# Resolve remaining duplicate EmployeeNumber values
# Each employee should be represented by a single record.
# If duplicates exist, retain the record with the most recent RecordID.

remaining_duplicates = df_clean['EmployeeNumber'].duplicated().sum()

print("Remaining duplicated EmployeeNumber values:", remaining_duplicates)

if remaining_duplicates > 0:
    df_clean = (
        df_clean
        .sort_values('RecordID')
        .drop_duplicates(subset='EmployeeNumber', keep='last')
    )

    print("Retained the most recent record for each EmployeeNumber.")
    print("Final dataset size:", len(df_clean))

Remaining duplicated EmployeeNumber values: 0


In [9]:
# Handle implausible age values
# Ages below 18 are considered implausible for this workforce and are
# temporarily converted to missing values for consistent imputation later.

df_clean.loc[df_clean['Age'] < 18, 'Age'] = np.nan

print("Converted implausible age values to missing values (NaN).")

Converted implausible age values to missing values (NaN).


In [10]:
# Cap potential outliers using the Interquartile Range (IQR) method
# The IQR approach provides a robust, data-driven way to reduce the influence
# of extreme values while preserving all observations in the dataset.

def cap_outliers_iqr(series, factor=1.5):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - factor * iqr
    upper_bound = q3 + factor * iqr

    capped_series = series.clip(lower=lower_bound, upper=upper_bound)

    return capped_series, lower_bound, upper_bound


# Apply IQR capping
df_clean['MonthlyIncome'], income_lower, income_upper = cap_outliers_iqr(
    df_clean['MonthlyIncome']
)

df_clean['DistanceFromHome'], distance_lower, distance_upper = cap_outliers_iqr(
    df_clean['DistanceFromHome']
)

print(f"MonthlyIncome bounds: [{income_lower:.0f}, {income_upper:.0f}]")
print(f"DistanceFromHome bounds: [{distance_lower:.0f}, {distance_upper:.0f}]")

MonthlyIncome bounds: [-5370, 16790]
DistanceFromHome bounds: [-16, 32]


In [11]:
# Impute the remaining missing values
# Numerical features are imputed using the median to reduce the influence of
# extreme values, while categorical features are imputed using the mode.

# Impute numerical features
numerical_cols = [
    'Age',
    'MonthlyIncome',
    'TotalWorkingYears',
    'YearsAtCompany'
]

for col in numerical_cols:
    median_value = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_value)
    print(f"{col}: missing values filled with median ({median_value})")


# Impute categorical features
categorical_cols = [
    'EducationField',
    'JobRole'
]

for col in categorical_cols:
    mode_value = df_clean[col].mode()[0]
    df_clean[col] = df_clean[col].fillna(mode_value)
    print(f"{col}: missing values filled with mode ({mode_value})")


print("\nRemaining missing values:")
print(df_clean.isna().sum().sum())

Age: missing values filled with median (36.0)
MonthlyIncome: missing values filled with median (4945.5)
TotalWorkingYears: missing values filled with median (10.0)
YearsAtCompany: missing values filled with median (5.0)
EducationField: missing values filled with mode (Life Sciences)
JobRole: missing values filled with mode (Sales Executive)

Remaining missing values:
0


In [12]:
# Restore the Age data type
# After missing value imputation, Age is converted back to an integer type
# because employee age is recorded as whole years.

df_clean['Age'] = df_clean['Age'].astype(int)

print(df_clean[['Age']].dtypes)

Age    int64
dtype: object


In [13]:
# Preview the cleaned dataset

print(f"Dataset shape after data cleaning: {df_clean.shape}")

display(df_clean.head())

Dataset shape after data cleaning: (1470, 33)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,RecordID
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,2,Female,94,3,2,Sales Executive,4,Single,4945.5,19479,8,Yes,11,3,1,0,8.0,0,1,6.0,4,0,5,100001
1,49,No,Travel_Frequently,279,Research and Development,8,1,Life Sciences,2,3,Male,61,2,2,Research Scientist,2,Married,4945.5,24907,1,No,23,4,4,1,10.0,3,3,10.0,7,1,7,100002
2,37,Yes,Travel_Rarely,1373,Research and Development,2,2,Other,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090.0,2396,6,Yes,15,3,2,0,7.0,3,3,0.0,0,0,0,100003
3,33,No,Travel_Frequently,1392,Research and Development,3,4,Life Sciences,5,4,Female,56,3,1,Research Scientist,3,Married,2909.0,23159,1,Yes,11,3,3,0,8.0,3,3,5.0,7,3,0,100004
4,27,No,Travel_Rarely,591,Research and Development,2,1,Medical,7,1,Male,40,3,1,Laboratory Technician,2,Married,3468.0,16632,9,No,12,3,4,1,6.0,3,3,2.0,2,2,2,100005


In [15]:
# Save the data after cleaning 
# Save the cleaned dataset as a new CSV file

output_file = r"D:\NTI-ML\nti-projects\employee-attrition-analysis\data\processed\IBM_HR_Attrition_Cleaned.csv"

df_clean.to_csv(output_file, index=False)

print(f"Cleaned dataset saved successfully as '{output_file}'.")

Cleaned dataset saved successfully as 'D:\NTI-ML\nti-projects\employee-attrition-analysis\data\processed\IBM_HR_Attrition_Cleaned.csv'.


#Data Quality Issues and Cleaning Summary

| Step | Data Quality Issue | Cleaning Technique | Rationale |
|:---:|---|---|---|
| **1** | Inconsistent categorical values (extra spaces, inconsistent capitalization) | `str.strip()` + `str.title()` | Standardize equivalent categories into a consistent representation. |
| **2** | Multiple representations of `BusinessTravel` categories | Mapping dictionary | Consolidate different spellings and formats into unified category labels. |
| **3** | Spelling errors and inconsistent department names (`Human Resourses`, `sales`) | Mapping dictionary | Correct known spelling mistakes and standardize department names. |
| **4** | Multiple variations of `JobRole` values (`Sales Ex.`, `Sales Executve`, etc.) | Mapping dictionary | Standardize job titles to a single consistent format. |
| **5** | Constant (zero-variance) features (`EmployeeCount`, `Over18`, `StandardHours`) | Feature removal | These variables contain no variation and provide no analytical or predictive value. |
| **6** | `ManagerNotes` contains approximately 60% missing values and unstructured text | Column removal | High missing rate and free-text content make the feature unsuitable for structured analysis. |
| **7** | Fully duplicated records | `drop_duplicates()` | Prevent duplicate observations from biasing descriptive statistics and machine learning models. |
| **8** | Remaining duplicated `EmployeeNumber` values | Retain the record with the highest `RecordID` | Ensure that each employee is represented by a single observation. |
| **9** | Implausible employee ages (<18 years) | Convert to `NaN` before imputation | Preserve the remaining employee information while handling implausible values consistently. |
| **10** | Potential outliers in `MonthlyIncome` and `DistanceFromHome` | IQR Capping (Winsorization) | Reduce the influence of extreme values while preserving all observations. |
| **11** | Missing values in numerical features | Median imputation | The median is robust to extreme values and preserves the central tendency of the data. |
| **12** | Missing values in categorical features | Mode imputation | Replace missing values with the most frequently occurring category. |
| **13** | `Age` stored as a floating-point variable after imputation | `astype(int)` | Restore the feature to its appropriate integer data type. |